# TranSTR Inference with W&B Weights

Notebook để **inference và visualize** model TranSTR:
- Load weights từ W&B
- Hiển thị 16 sampled frames
- Visualize Temporal Rationalization (frames được chọn)
- Visualize Spatial Rationalization (objects được chọn)
- Hiển thị Q&A và prediction cho **TẤT CẢ** các dạng câu hỏi

In [ ]:
# CELL 1: Setup & Imports
import os, sys, json, pickle
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import cv2
from einops import rearrange

!pip install -q wandb
import wandb
print('Imports OK')

In [ ]:
# CELL 2: Configuration
print('=== CELL 2: Configuration ===')

WANDB_API_KEY = 'YOUR_WANDB_API_KEY_HERE'
WANDB_PROJECT = 'transtr-causalvid'
WANDB_ARTIFACT = 'best-model:latest'

VIT_FEATURE_PATH = '/kaggle/input/vit-features-full-merged'
OBJ_FEATURE_PATH = '/kaggle/input/object-detection-causal-full'
ANNOTATION_PATH = '/kaggle/input/text-annotation/QA'
SPLIT_DIR = '/kaggle/input/casual-vid-data-split/split'
VIDEO_ROOT = '/kaggle/input/causalvid-videos'

ALL_QUESTION_TYPES = ['descriptive', 'explanatory', 'predictive', 'predictive_reason', 'counterfactual', 'counterfactual_reason']

class Config:
    topK_frame = 16
    objs = 20
    frames = 16
    select_frames = 5
    topK_obj = 12
    frame_feat_dim = 1024
    obj_feat_dim = 2053
    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3
    text_encoder_type = 'microsoft/deberta-base'
    freeze_text_encoder = False
    n_query = 5
    hard_eval = True

args = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# CELL 3: Load Model from W&B
print('=== CELL 3: Load Model ===')

from networks.model import VideoQAmodel
from utils.util import set_seed
set_seed(999)

wandb.login(key=WANDB_API_KEY)
api = wandb.Api()
artifact = api.artifact(f'{WANDB_PROJECT}/{WANDB_ARTIFACT}')
artifact_dir = artifact.download()
print(f'Downloaded to: {artifact_dir}')

ckpt_path = None
for f in os.listdir(artifact_dir):
    if f.endswith('.ckpt') or f.endswith('.pt') or f.endswith('.pth'):
        ckpt_path = os.path.join(artifact_dir, f)
        break
print(f'Checkpoint: {ckpt_path}')

cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
cfg['topK_frame'] = args.select_frames

model = VideoQAmodel(**cfg)
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.to(device)
model.eval()
print('Model loaded!')

In [ ]:
# CELL 4: Helper Functions
print('=== CELL 4: Helper Functions ===')

def find_video_path(video_id, video_root):
    video_root = Path(video_root)
    for ext in ['.mp4', '.avi', '.mkv', '.mov', '.webm']:
        path = video_root / f"{video_id}{ext}"
        if path.exists(): return path
        for subpath in video_root.rglob(f"{video_id}{ext}"): return subpath
    return None

def load_video_frames(video_path, num_frames=16):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total - 1, num_frames, dtype=int)
    frames, frame_indices = [], []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if ok:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            frame_indices.append(idx)
    cap.release()
    return frames, frame_indices, total

def load_qa_data(video_id, annotation_path):
    vp = os.path.join(annotation_path, video_id)
    qa_data = {}
    text_file, ans_file = os.path.join(vp, 'text.json'), os.path.join(vp, 'answer.json')
    if os.path.exists(text_file) and os.path.exists(ans_file):
        with open(text_file, 'r', encoding='utf-8') as f: text_data = json.load(f)
        with open(ans_file, 'r', encoding='utf-8') as f: ans_data = json.load(f)
        for qtype in ['descriptive', 'explanatory', 'predictive', 'counterfactual']:
            if qtype in text_data and qtype in ans_data:
                qa_data[qtype] = {'question': text_data[qtype].get('question', ''), 'choices': text_data[qtype].get('answer', []), 'correct_idx': ans_data[qtype].get('answer', 0)}
                if 'reason' in text_data[qtype]:
                    qa_data[f'{qtype}_reason'] = {'question': 'Why?', 'choices': text_data[qtype].get('reason', []), 'correct_idx': ans_data[qtype].get('reason', 0)}
    return qa_data

print('Helper functions defined')

In [ ]:
# CELL 5: Inference Function
print('=== CELL 5: Inference Function ===')
from networks.topk import HardtopK
from utils.util import transform_bb

@torch.no_grad()
def inference_with_attention(model, video_id, qtype, device):
    ff = torch.load(os.path.join(VIT_FEATURE_PATH, f'{video_id}.pt'), weights_only=True)
    if isinstance(ff, np.ndarray): ff = torch.from_numpy(ff)
    ff = ff.float()
    nf = ff.shape[0]
    if nf > args.topK_frame:
        indices = np.linspace(0, nf - 1, args.topK_frame).astype(int)
        ff = ff[indices]
    elif nf < args.topK_frame:
        ff = torch.cat([ff, torch.zeros(args.topK_frame - nf, ff.shape[1])], 0)
    
    obj_pkl = None
    for subdir in os.listdir(OBJ_FEATURE_PATH):
        subdir_path = os.path.join(OBJ_FEATURE_PATH, subdir)
        if os.path.isdir(subdir_path):
            pkl_path = os.path.join(subdir_path, f'{video_id}.pkl')
            if os.path.exists(pkl_path): obj_pkl = pkl_path; break
    
    objs, obj_bboxes = [], []
    if obj_pkl:
        with open(obj_pkl, 'rb') as f: data = pickle.load(f)
        feats, bboxes = np.array(data.get('features')), np.array(data.get('bboxes'))
        num_frames = feats.shape[0]
        indices = np.linspace(0, num_frames - 1, args.topK_frame).astype(int) if num_frames > args.topK_frame else range(num_frames)
        for i in indices:
            feat, bbox = torch.from_numpy(feats[i]).float(), torch.from_numpy(bboxes[i]).float()
            if feat.shape[0] > args.objs: feat, bbox = feat[:args.objs], bbox[:args.objs]
            elif feat.shape[0] < args.objs:
                p = args.objs - feat.shape[0]
                feat = torch.cat([feat, torch.zeros(p, feat.shape[1])], 0)
                bbox = torch.cat([bbox, torch.zeros(p, bbox.shape[1])], 0)
            bb = torch.from_numpy(transform_bb(bbox.numpy(), 640, 480)).float()
            objs.append(torch.cat([feat, bb], -1))
            obj_bboxes.append(bbox.numpy())
    while len(objs) < args.topK_frame:
        objs.append(torch.zeros(args.objs, 2053))
        obj_bboxes.append(np.zeros((args.objs, 4)))
    of = torch.stack(objs)
    
    qa_data = load_qa_data(video_id, ANNOTATION_PATH)
    if qtype not in qa_data: raise ValueError(f'{qtype} not found for {video_id}')
    qa = qa_data[qtype]
    qns, choices, correct_idx = qa['question'], qa['choices'], qa['correct_idx']
    ans_word = [f"{qns} [SEP] {c}" for c in choices]
    
    ff, of = ff.unsqueeze(0).to(device), of.unsqueeze(0).to(device)
    B, F, O = of.size()[:3]
    frame_feat = model.frame_resize(ff)
    q_local, q_mask = model.forward_text([qns], device)
    frame_mask = torch.ones(B, F).bool().to(device)
    frame_local, frame_att = model.frame_decoder(frame_feat, q_local, memory_key_padding_mask=q_mask, query_pos=model.pos_encoder_1d(frame_mask, model.d_model), output_attentions=True)
    idx_frame = rearrange(HardtopK(frame_att.flatten(1,2), model.frame_topK), 'b (f q) k -> b f q k', f=F).sum(-2)
    selected_frame_scores = idx_frame[0].sum(-1).cpu().numpy()
    frame_local = (frame_local.transpose(1,2) @ idx_frame).transpose(1,2)
    obj_feat = (of.flatten(-2,-1).transpose(1,2) @ idx_frame).transpose(1,2).view(B, model.frame_topK, O, -1)
    obj_local = model.obj_resize(obj_feat)
    q_local_rep = q_local.repeat_interleave(model.frame_topK, dim=0)
    q_mask_rep = q_mask.repeat_interleave(model.frame_topK, dim=0) if q_mask is not None else None
    obj_local, obj_att = model.obj_decoder(obj_local.flatten(0,1), q_local_rep, memory_key_padding_mask=q_mask_rep, output_attentions=True)
    idx_obj = rearrange(HardtopK(obj_att.flatten(1,2), model.obj_topK), 'b (o q) k -> b o q k', o=O).sum(-2)
    selected_obj_scores = idx_obj.view(model.frame_topK, O, -1).sum(-1).cpu().numpy()
    out = model(ff, of, qns, ans_word)
    pred = out.argmax(-1).item()
    probs = torch.softmax(out, dim=-1)[0].cpu().numpy()
    return {'prediction': pred, 'probs': probs, 'correct_idx': correct_idx, 'question': qns, 'choices': choices, 'frame_scores': selected_frame_scores, 'obj_scores': selected_obj_scores, 'obj_bboxes': obj_bboxes}

print('Inference function defined')

In [ ]:
# CELL 6: Visualization
print('=== CELL 6: Visualization ===')

def visualize_inference(video_id, qtype='descriptive', show_frames=True):
    print(f'\n{"="*60}')
    print(f'Video: {video_id} | Question Type: {qtype}')
    print('='*60)
    video_path = find_video_path(video_id, VIDEO_ROOT)
    if video_path is None: print(f'Video not found: {video_id}'); return None
    frames, frame_indices, total_frames = load_video_frames(video_path, args.topK_frame)
    print(f'Loaded {len(frames)} frames from {total_frames} total')
    try: result = inference_with_attention(model, video_id, qtype, device)
    except Exception as e: print(f'Inference error: {e}'); return None
    
    if show_frames:
        # Fig 1: 16 Frames
        fig1, axes = plt.subplots(2, 8, figsize=(20, 6))
        fig1.suptitle(f'16 Sampled Frames - {video_id} - {qtype}', fontsize=14, fontweight='bold')
        for i, ax in enumerate(axes.flatten()):
            if i < len(frames):
                ax.imshow(frames[i])
                score = result['frame_scores'][i] if i < len(result['frame_scores']) else 0
                ax.set_title(f'F{i} ({score:.2f})', color='green' if score > 0.5 else 'red', fontsize=10)
            ax.axis('off')
        plt.tight_layout(); plt.show()
        
        # Fig 2: Temporal
        fig2, ax = plt.subplots(figsize=(12, 4))
        scores = result['frame_scores']
        colors = ['green' if s > 0.5 else 'lightgray' for s in scores]
        bars = ax.bar(range(len(scores)), scores, color=colors, edgecolor='black')
        ax.axhline(y=0.5, color='red', linestyle='--', label='Threshold')
        ax.set_xlabel('Frame Index'); ax.set_ylabel('Score')
        ax.set_title(f'Temporal Rationalization - Top {args.select_frames} frames')
        selected_indices = np.argsort(scores)[-args.select_frames:]
        for idx in selected_indices: bars[idx].set_edgecolor('blue'); bars[idx].set_linewidth(3)
        plt.tight_layout(); plt.show()
        
        # Fig 3: Spatial
        selected_frames = sorted(selected_indices)
        fig3, axes = plt.subplots(1, len(selected_frames), figsize=(4*len(selected_frames), 4))
        if len(selected_frames) == 1: axes = [axes]
        fig3.suptitle('Spatial Rationalization - Selected Frames', fontsize=12, fontweight='bold')
        for ax_idx, frame_idx in enumerate(selected_frames):
            if frame_idx < len(frames):
                ax = axes[ax_idx]; ax.imshow(frames[frame_idx])
                if ax_idx < len(result['obj_scores']):
                    obj_scores, bboxes = result['obj_scores'][ax_idx], result['obj_bboxes'][frame_idx]
                    top_obj_indices = np.argsort(obj_scores)[-args.topK_obj:]
                    for obj_idx in top_obj_indices:
                        if obj_idx < len(bboxes) and obj_scores[obj_idx] > 0:
                            x1, y1, x2, y2 = bboxes[obj_idx][:4]
                            ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='lime', linewidth=2))
                ax.set_title(f'Frame {frame_idx}'); ax.axis('off')
        plt.tight_layout(); plt.show()
    
    # Fig 4: Q&A
    fig4, ax = plt.subplots(figsize=(12, 5)); ax.axis('off')
    text = f"Question: {result['question']}\n\n"
    for i, choice in enumerate(result['choices']):
        prefix = ''
        if i == result['correct_idx']: prefix = 'CORRECT '
        if i == result['prediction']: prefix += '>>> '
        prob = result['probs'][i] * 100
        text += f"{prefix}({i}) {choice} [{prob:.1f}%]\n"
    is_correct = result['prediction'] == result['correct_idx']
    text += f"\nPrediction: {result['prediction']} | GT: {result['correct_idx']} | {'CORRECT' if is_correct else 'WRONG'}"
    ax.text(0.05, 0.5, text, transform=ax.transAxes, fontsize=11, verticalalignment='center', fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightgreen' if is_correct else 'lightcoral', alpha=0.5))
    ax.set_title(f'Q&A Results - {qtype.upper()}', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
    return result

print('Visualization defined')

In [ ]:
# CELL 7: Run ALL Question Types for Single Video
print('=== CELL 7: Run Inference - All Question Types ===')

def visualize_video_all_questions(video_id, show_frames_once=True):
    print(f'\n{"#"*70}')
    print(f'# VIDEO: {video_id} - ALL QUESTION TYPES')
    print('#'*70)
    qa_data = load_qa_data(video_id, ANNOTATION_PATH)
    print(f'Available: {list(qa_data.keys())}')
    results = {}
    for i, qtype in enumerate(ALL_QUESTION_TYPES):
        if qtype in qa_data:
            print(f'\n>>> [{i+1}/{len(ALL_QUESTION_TYPES)}] {qtype.upper()}')
            show = show_frames_once and (i == 0)
            try: results[qtype] = visualize_inference(video_id, qtype, show_frames=show)
            except Exception as e: print(f'Error: {e}'); results[qtype] = None
        else: print(f'Skipping {qtype}: not available')
    
    print(f'\n{"-"*60}')
    print(f'SUMMARY - {video_id}')
    print('-'*60)
    correct, total = 0, 0
    for qtype, res in results.items():
        if res:
            ok = res['prediction'] == res['correct_idx']
            print(f"{'O' if ok else 'X'} {qtype:25} | Pred: {res['prediction']} | GT: {res['correct_idx']}")
            total += 1; correct += int(ok)
    print(f'\nAccuracy: {correct}/{total} ({correct/total*100:.1f}%)' if total > 0 else 'No results')
    return results

# ============================================
# SPECIFY VIDEO ID HERE
# ============================================
VIDEO_ID = 'YOUR_VIDEO_ID_HERE'
all_results = visualize_video_all_questions(VIDEO_ID)

In [ ]:
# CELL 8: Batch Test - All Videos, All Question Types
print('=== CELL 8: Test Set - All Question Types ===')

test_pkl = os.path.join(SPLIT_DIR, 'test.pkl')
if os.path.exists(test_pkl):
    with open(test_pkl, 'rb') as f: test_vids = list(pickle.load(f))[:5]
    print(f'Testing {len(test_vids)} videos x {len(ALL_QUESTION_TYPES)} types')
    
    all_results = {}
    type_acc = {q: {'c': 0, 't': 0} for q in ALL_QUESTION_TYPES}
    
    for vid in test_vids:
        try:
            results = visualize_video_all_questions(vid, show_frames_once=True)
            all_results[vid] = results
            for q, r in results.items():
                if r:
                    type_acc[q]['t'] += 1
                    if r['prediction'] == r['correct_idx']: type_acc[q]['c'] += 1
        except Exception as e: print(f'Error {vid}: {e}')
    
    print(f'\n{"="*70}')
    print('OVERALL SUMMARY')
    print('='*70)
    tc, tt = 0, 0
    for q in ALL_QUESTION_TYPES:
        c, t = type_acc[q]['c'], type_acc[q]['t']
        print(f"{q:25} | {c:3}/{t:3} | {c/t*100:.1f}%" if t > 0 else f"{q:25} | 0/0")
        tc += c; tt += t
    print('-'*70)
    print(f"{'TOTAL':25} | {tc:3}/{tt:3} | {tc/tt*100:.1f}%" if tt > 0 else 'No data')
else:
    print('Test split not found')

## CELL 9: Export Correct/Incorrect Predictions to CSV

Chạy inference trên **toàn bộ test set** cho cả 6 loại câu hỏi, lưu kết quả vào CSV:
- `results_correct_all.csv` & `results_incorrect_all.csv` (tổng hợp)
- `results_correct_{qtype}.csv` & `results_incorrect_{qtype}.csv` (theo từng loại)

**Columns:** `video_id`, `question_type`, `question`, `a0..a4`, `correct_idx`, `predicted_idx`,
`is_correct`, `confidence`, `prob_a0..prob_a4`, `predicted_answer`, `correct_answer`

CSV files được thiết kế để dễ dàng visualize bằng pandas/seaborn/plotly.

In [ ]:
# CELL 9: Export Correct/Incorrect Predictions to CSV with Probabilities
import pandas as pd
from tqdm import tqdm
import traceback

print('=== CELL 9: Export Predictions to CSV ===')

# ============================================
# CONFIGURATION
# ============================================
OUTPUT_DIR = './inference_results'  # Thư mục lưu CSV
MAX_TEST_VIDEOS = None  # None = full test set, hoặc set số nhỏ để test nhanh
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load test video IDs
test_pkl = os.path.join(SPLIT_DIR, 'test.pkl')
test_txt = os.path.join(SPLIT_DIR, 'test.txt')

test_vids = []
if os.path.exists(test_pkl):
    with open(test_pkl, 'rb') as f:
        test_vids = list(pickle.load(f))
    print(f'Loaded {len(test_vids)} test videos from pkl')
elif os.path.exists(test_txt):
    with open(test_txt, 'r') as f:
        test_vids = [l.strip() for l in f if l.strip()]
    print(f'Loaded {len(test_vids)} test videos from txt')
else:
    raise FileNotFoundError('Test split file not found!')

if MAX_TEST_VIDEOS:
    test_vids = test_vids[:MAX_TEST_VIDEOS]
    print(f'Limited to {MAX_TEST_VIDEOS} videos for testing')

print(f'Running inference on {len(test_vids)} videos x {len(ALL_QUESTION_TYPES)} question types...\n')

# ============================================
# BATCH INFERENCE
# ============================================
all_records = []
type_stats = {q: {'correct': 0, 'total': 0} for q in ALL_QUESTION_TYPES}
errors = []

for vid_idx, video_id in enumerate(tqdm(test_vids, desc='Inference')):
    qa_data = load_qa_data(video_id, ANNOTATION_PATH)
    
    for qtype in ALL_QUESTION_TYPES:
        if qtype not in qa_data:
            continue
        
        try:
            result = inference_with_attention(model, video_id, qtype, device)
            
            pred_idx = result['prediction']
            correct_idx = result['correct_idx']
            probs = result['probs']
            choices = result['choices']
            question = result['question']
            is_correct = (pred_idx == correct_idx)
            confidence = float(probs[pred_idx])
            
            # Build record
            record = {
                'video_id': video_id,
                'question_type': qtype,
                'question': question,
                'correct_idx': correct_idx,
                'predicted_idx': pred_idx,
                'is_correct': is_correct,
                'confidence': round(confidence, 6),
                'correct_answer': choices[correct_idx] if correct_idx < len(choices) else '',
                'predicted_answer': choices[pred_idx] if pred_idx < len(choices) else '',
            }
            
            # Add each answer choice and its probability
            for i in range(5):  # max 5 answer choices
                record[f'a{i}'] = choices[i] if i < len(choices) else ''
                record[f'prob_a{i}'] = round(float(probs[i]), 6) if i < len(probs) else 0.0
            
            all_records.append(record)
            
            type_stats[qtype]['total'] += 1
            if is_correct:
                type_stats[qtype]['correct'] += 1
                
        except Exception as e:
            errors.append({'video_id': video_id, 'qtype': qtype, 'error': str(e)})

print(f'\nInference complete: {len(all_records)} predictions, {len(errors)} errors')

# ============================================
# CREATE DATAFRAME & SAVE CSV FILES
# ============================================
df_all = pd.DataFrame(all_records)

# Define column order for readability
col_order = [
    'video_id', 'question_type', 'question',
    'a0', 'a1', 'a2', 'a3', 'a4',
    'correct_idx', 'predicted_idx', 'is_correct', 'confidence',
    'prob_a0', 'prob_a1', 'prob_a2', 'prob_a3', 'prob_a4',
    'correct_answer', 'predicted_answer'
]
col_order = [c for c in col_order if c in df_all.columns]
df_all = df_all[col_order]

df_correct = df_all[df_all['is_correct'] == True].copy()
df_incorrect = df_all[df_all['is_correct'] == False].copy()

# --- Save combined files ---
path_correct_all = os.path.join(OUTPUT_DIR, 'results_correct_all.csv')
path_incorrect_all = os.path.join(OUTPUT_DIR, 'results_incorrect_all.csv')
df_correct.to_csv(path_correct_all, index=False, encoding='utf-8-sig')
df_incorrect.to_csv(path_incorrect_all, index=False, encoding='utf-8-sig')

# --- Save per question type ---
for qtype in ALL_QUESTION_TYPES:
    df_q = df_all[df_all['question_type'] == qtype]
    if len(df_q) == 0:
        continue
    df_q_correct = df_q[df_q['is_correct'] == True]
    df_q_incorrect = df_q[df_q['is_correct'] == False]
    
    df_q_correct.to_csv(os.path.join(OUTPUT_DIR, f'results_correct_{qtype}.csv'), index=False, encoding='utf-8-sig')
    df_q_incorrect.to_csv(os.path.join(OUTPUT_DIR, f'results_incorrect_{qtype}.csv'), index=False, encoding='utf-8-sig')

# --- Save errors log if any ---
if errors:
    pd.DataFrame(errors).to_csv(os.path.join(OUTPUT_DIR, 'inference_errors.csv'), index=False, encoding='utf-8-sig')

# ============================================
# PRINT SUMMARY
# ============================================
print(f'\n{"="*70}')
print(f'{"QUESTION TYPE":<25} | {"CORRECT":>8} | {"TOTAL":>8} | {"ACC":>8}')
print(f'{"-"*70}')
total_c, total_t = 0, 0
for qtype in ALL_QUESTION_TYPES:
    c = type_stats[qtype]['correct']
    t = type_stats[qtype]['total']
    acc = f'{c/t*100:.1f}%' if t > 0 else 'N/A'
    print(f'{qtype:<25} | {c:>8} | {t:>8} | {acc:>8}')
    total_c += c
    total_t += t
overall_acc = f'{total_c/total_t*100:.1f}%' if total_t > 0 else 'N/A'
print(f'{"-"*70}')
print(f'{"OVERALL":<25} | {total_c:>8} | {total_t:>8} | {overall_acc:>8}')
print(f'{"="*70}')

print(f'\n📁 Saved to: {os.path.abspath(OUTPUT_DIR)}')
print(f'   ✅ Correct:   {len(df_correct)} samples -> results_correct_all.csv')
print(f'   ❌ Incorrect: {len(df_incorrect)} samples -> results_incorrect_all.csv')
print(f'   📊 Per-type files: results_{{correct|incorrect}}_{{qtype}}.csv')
if errors:
    print(f'   ⚠️  Errors: {len(errors)} -> inference_errors.csv')
print(f'\n💡 Tip: Load with pd.read_csv() and use seaborn/plotly for visualization')

### Quick Visualization Examples

Các ví dụ nhanh để visualize kết quả từ CSV vừa export.

In [ ]:
# CELL 10: Quick Visualization from CSV
import matplotlib.pyplot as plt
import numpy as np

print('=== CELL 10: Quick Visualization ===')

# Load combined results
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'results_correct_all.csv'))
df_wrong = pd.read_csv(os.path.join(OUTPUT_DIR, 'results_incorrect_all.csv'))
df_all_viz = pd.concat([df, df_wrong], ignore_index=True)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('TranSTR Inference Analysis', fontsize=16, fontweight='bold')

# --- Plot 1: Accuracy per Question Type ---
ax = axes[0, 0]
acc_by_type = df_all_viz.groupby('question_type')['is_correct'].mean() * 100
acc_by_type = acc_by_type.reindex(ALL_QUESTION_TYPES).dropna()
colors_acc = ['#2ecc71' if v >= 50 else '#e74c3c' for v in acc_by_type.values]
bars = ax.barh(range(len(acc_by_type)), acc_by_type.values, color=colors_acc, edgecolor='white', height=0.6)
ax.set_yticks(range(len(acc_by_type)))
ax.set_yticklabels(acc_by_type.index, fontsize=10)
ax.set_xlabel('Accuracy (%)')
ax.set_title('Accuracy per Question Type')
ax.set_xlim(0, 100)
for i, (v, bar) in enumerate(zip(acc_by_type.values, bars)):
    ax.text(v + 1, i, f'{v:.1f}%', va='center', fontweight='bold', fontsize=10)
ax.axvline(x=50, color='gray', linestyle='--', alpha=0.5, label='50% baseline')
ax.legend()

# --- Plot 2: Confidence Distribution (Correct vs Incorrect) ---
ax = axes[0, 1]
if len(df) > 0:
    ax.hist(df['confidence'], bins=30, alpha=0.6, color='#2ecc71',
            label=f'Correct (n={len(df)})', density=True)
if len(df_wrong) > 0:
    ax.hist(df_wrong['confidence'], bins=30, alpha=0.6, color='#e74c3c',
            label=f'Incorrect (n={len(df_wrong)})', density=True)
ax.set_xlabel('Confidence (softmax probability)')
ax.set_ylabel('Density')
ax.set_title('Confidence Distribution')
ax.legend()

# --- Plot 3: Confusion Matrix (Predicted vs Correct Index) ---
ax = axes[1, 0]
from collections import Counter
confusion = Counter(zip(df_all_viz['correct_idx'], df_all_viz['predicted_idx']))
max_idx = max(max(df_all_viz['correct_idx']), max(df_all_viz['predicted_idx'])) + 1
matrix = np.zeros((max_idx, max_idx))
for (gt, pred), count in confusion.items():
    matrix[gt][pred] = count
im = ax.imshow(matrix, cmap='YlOrRd', interpolation='nearest')
ax.set_xlabel('Predicted Index')
ax.set_ylabel('Correct Index')
ax.set_title('Answer Index Confusion Matrix')
for i in range(max_idx):
    for j in range(max_idx):
        ax.text(j, i, int(matrix[i][j]), ha='center', va='center', fontsize=10,
                color='white' if matrix[i][j] > matrix.max()/2 else 'black')
plt.colorbar(im, ax=ax)

# --- Plot 4: Probability Boxplot per Answer Choice (Incorrect only) ---
ax = axes[1, 1]
if len(df_wrong) > 0:
    prob_cols = [f'prob_a{i}' for i in range(5) if f'prob_a{i}' in df_wrong.columns]
    prob_data = df_wrong[prob_cols].values
    bp = ax.boxplot([prob_data[:, i] for i in range(len(prob_cols))],
                    positions=range(len(prob_cols)), widths=0.6, patch_artist=True)
    colors_box = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c', '#9b59b6']
    for patch, color in zip(bp['boxes'], colors_box[:len(prob_cols)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_xticklabels([f'A{i}' for i in range(len(prob_cols))])
    ax.set_xlabel('Answer Choice')
    ax.set_ylabel('Probability')
ax.set_title('Prob Distribution per Choice (Incorrect Only)')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'analysis_overview.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'\n📊 Saved analysis plot to {OUTPUT_DIR}/analysis_overview.png')